# Dafne + MedSAM Thigh Segmentation

Runs the Dafne Thigh model slice-by-slice, then refines each muscle mask with MedSAM using the Dafne bounding box as the prompt. No manual point picking required.

MedSAM embedding is computed **once per slice** and reused for all 24 muscles.

**Kernel:** `dafne_clean`

In [10]:
import glob
import os
import sys
import numpy as np
import SimpleITK as sitk
import torch
from skimage import transform
from dafne_dl import DynamicDLModel
from dafne.config import GlobalConfig
from dafne.utils.sam_mask_refine import (
    load_sam,
    medsam_inference,
    enlarge_bounding_box,
    determine_device,
)

In [11]:
# --- paths ---
DEVICE      = determine_device()
MODEL_PATH  = "dafne_thigh_results/model_used/Thigh_1774532147.model"
IMAGE_GLOB  = "myosegmenTUM/*/ImageData/*FATFRACTION/*FATFRACTION_stack*.nii"
OUTPUT_DIR  = "dafne_medsam_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Device:", DEVICE)

SAM loaded on CPU
Device: cpu


In [12]:
# load Dafne model
dafne_model = DynamicDLModel.Load(open(MODEL_PATH, "rb"))
print("Dafne model loaded:", MODEL_PATH)

Dafne model loaded: dafne_thigh_results/model_used/Thigh_1774532147.model


In [13]:
# load MedSAM model once — downloads medsam_vit_b.pth (~375 MB) if not already present
GlobalConfig['SAM_MODEL'] = 'Med Sam'
sam_model = load_sam('Med Sam')
sam_model.eval()
print("MedSAM loaded on", DEVICE)

SAM loaded on CPU
MedSAM loaded on cpu


In [14]:
image_files = sorted(glob.glob(IMAGE_GLOB))
print(f"Found {len(image_files)} images:")
for p in image_files:
    print(" ", p)

Found 54 images:
  myosegmenTUM\HV001_1\ImageData\HV001_1_FATFRACTION\HV001_1_FATFRACTION_stack1.nii
  myosegmenTUM\HV001_1\ImageData\HV001_1_FATFRACTION\HV001_1_FATFRACTION_stack2.nii
  myosegmenTUM\HV001_2\ImageData\HV001_2_FATFRACTION\HV001_2_FATFRACTION_stack1.nii
  myosegmenTUM\HV001_2\ImageData\HV001_2_FATFRACTION\HV001_2_FATFRACTION_stack2.nii
  myosegmenTUM\HV001_3\ImageData\HV001_3_FATFRACTION\HV001_3_FATFRACTION_stack1.nii
  myosegmenTUM\HV001_3\ImageData\HV001_3_FATFRACTION\HV001_3_FATFRACTION_stack2.nii
  myosegmenTUM\HV002_1\ImageData\HV002_1_FATFRACTION\HV002_1_FATFRACTION_stack1.nii
  myosegmenTUM\HV002_1\ImageData\HV002_1_FATFRACTION\HV002_1_FATFRACTION_stack2.nii
  myosegmenTUM\HV002_2\ImageData\HV002_2_FATFRACTION\HV002_2_FATFRACTION_stack1.nii
  myosegmenTUM\HV002_2\ImageData\HV002_2_FATFRACTION\HV002_2_FATFRACTION_stack2.nii
  myosegmenTUM\HV002_3\ImageData\HV002_3_FATFRACTION\HV002_3_FATFRACTION_stack1.nii
  myosegmenTUM\HV002_3\ImageData\HV002_3_FATFRACTION\HV002_

In [ ]:
# run Dafne + MedSAM refinement slice-by-slice
for nii_path in image_files:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    out_path = os.path.join(OUTPUT_DIR, f"{stem}_dafne_medsam.npz")

    if os.path.exists(out_path):
        print(f"Skipping (already done): {out_path}")
        continue

    print(f"\nProcessing: {nii_path}")
    img_sitk  = sitk.ReadImage(nii_path)
    img_array = sitk.GetArrayFromImage(img_sitk).astype(float)  # (slices, H, W)
    spacing   = img_sitk.GetSpacing()
    resolution = [spacing[0], spacing[1]]
    H, W = img_array.shape[1], img_array.shape[2]
    print(f"  Shape: {img_array.shape}  Resolution: {resolution}")

    all_masks = {}  # {muscle_name: 3D uint8 array}

    for slice_idx in range(img_array.shape[0]):
        slice_2d = img_array[slice_idx]

        # --- Dafne segmentation ---
        dafne_out = dafne_model({
            "image": slice_2d,
            "resolution": resolution,
            "split_laterality": True,
            "classification": "Thigh",
        })

        # --- MedSAM embedding (once per slice) ---
        img_norm = slice_2d * 255.0 / (slice_2d.max() + 1e-8)
        img_3c   = np.repeat(img_norm[:, :, None], 3, axis=-1)
        img_1024 = transform.resize(
            img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True
        ).astype(np.uint8)
        img_1024 = (img_1024 - img_1024.min()) / np.clip(
            img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None
        )
        img_tensor = torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            image_embedding = sam_model.image_encoder(img_tensor)

        # --- refine each muscle mask ---
        for muscle_name, mask in dafne_out.items():
            mask_arr = np.asarray(mask, dtype=np.uint8)

            if mask_arr.any():
                bbox     = enlarge_bounding_box(mask_arr)              # [min_col, min_row, max_col, max_row]
                box_1024 = bbox / np.array([W, H, W, H]) * 1024
                box_1024 = box_1024[None, None, :]                     # (1, 1, 4)
                refined  = medsam_inference(sam_model, image_embedding, box_1024, H, W)
            else:
                refined = mask_arr

            if muscle_name not in all_masks:
                all_masks[muscle_name] = np.zeros(img_array.shape, dtype=np.uint8)
            all_masks[muscle_name][slice_idx] = refined.astype(np.uint8)

        if (slice_idx + 1) % 5 == 0 or slice_idx == img_array.shape[0] - 1:
            print(f"  slice {slice_idx + 1}/{img_array.shape[0]} done")

    np.savez_compressed(out_path, **all_masks)
    print(f"  Saved → {out_path}")
    print(f"  Muscles: {list(all_masks.keys())}")

print("\nAll done.")

Skipping (already done): dafne_medsam_results\HV001_1_FATFRACTION_stack1_dafne_medsam.npz
Skipping (already done): dafne_medsam_results\HV001_1_FATFRACTION_stack2_dafne_medsam.npz
Skipping (already done): dafne_medsam_results\HV001_2_FATFRACTION_stack1_dafne_medsam.npz
Skipping (already done): dafne_medsam_results\HV001_2_FATFRACTION_stack2_dafne_medsam.npz
Skipping (already done): dafne_medsam_results\HV001_3_FATFRACTION_stack1_dafne_medsam.npz
Skipping (already done): dafne_medsam_results\HV001_3_FATFRACTION_stack2_dafne_medsam.npz
Skipping (already done): dafne_medsam_results\HV002_1_FATFRACTION_stack1_dafne_medsam.npz
Skipping (already done): dafne_medsam_results\HV002_1_FATFRACTION_stack2_dafne_medsam.npz
Skipping (already done): dafne_medsam_results\HV002_2_FATFRACTION_stack1_dafne_medsam.npz
Skipping (already done): dafne_medsam_results\HV002_2_FATFRACTION_stack2_dafne_medsam.npz

Processing: myosegmenTUM\HV002_3\ImageData\HV002_3_FATFRACTION\HV002_3_FATFRACTION_stack1.nii
  Sha

In [ ]:
# sanity check — reload one result
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.npz")))
if results:
    sample = np.load(results[0])
    print("Sample file:", results[0])
    for name in sample.files:
        arr = sample[name]
        print(f"  {name}: shape={arr.shape}  positive voxels={arr.sum()}")